In [4]:
from pathlib import Path
import pandas as pd

In [ ]:
print(df.columns.tolist())

NameError: name 'df' is not defined

In [7]:
df = pd.read_csv('C:\\Users\\samue\\OneDrive\\Desktop\\tcs\\Auditor-Project\\data\\raw\\LoanData_Bondora.csv', low_memory=False)

df['ano'] = pd.to_datetime(df['LoanDate'], errors='coerce').dt.year
df['default'] = df['DefaultDate'].notna().astype(int)
df['encerrado'] = (df['Status'] == 'Repaid') | (df['default'] == 1)

resumo = df.groupby('ano').agg(
    contratos=('LoanId', 'size'),
    pct_encerrados=('encerrado', 'mean'),
    pct_defaults=('default', 'mean')
)

print(resumo.round(3))


      contratos  pct_encerrados  pct_defaults
ano                                          
2009        665           1.000         0.389
2010       1157           1.000         0.378
2011        451           1.000         0.277
2012        457           0.998         0.208
2013       2512           0.988         0.353
2014       7455           0.982         0.549
2015       8046           0.966         0.604
2016      10514           0.942         0.594
2017      17933           0.879         0.587
2018      25103           0.753         0.487
2019      55839           0.659         0.459
2020      27519           0.381         0.215
2021      21584           0.065         0.006


In [ ]:
base = df[df['ano'].between(2009, 2016)& df['encerrado']].copy()
print('Contratos no recorte:', len(base))
print('Taxa de default', round(base['default'].mean(), 3))

print('\n== Contratos por país e ano ==')
print(pd.crosstab(base['ano'], base['Country'], margins=True))

print('n==Taxa de default por país ==')
print(base.groupby('Country')['default'].agg(['size', 'mean']).round(3))

por_cliente = base.groupby('UserName').size()
print('\n== Contratos por cliente ==')
print('Clientes distintos:', por_cliente.size)
print(por_cliente.value_counts().sort_index(). head(10))
print('% de contratos por cliente com 1 contrato:', round(por_cliente[por_cliente > 1].sum() / len(base), 3))

treino_cli = set(base.loc[base['ano'] <= 2015, 'UserName'])
teste_cli = set(base.loc[base['ano'] == 2016, 'UserName'])
print('\nClientes distintos no treino e no teste: ', len(treino_cli & teste_cli)) 



Contratos no recorte: 30210
Taxa de default 0.563

== Contratos por país e ano ==
Country     EE    ES    FI   SK    All
ano                                   
2009       665     0     0    0    665
2010      1157     0     0    0   1157
2011       451     0     0    0    451
2012       456     0     0    0    456
2013      2255    61   165    0   2481
2014      4044  1603  1386  291   7324
2015      3225  2184  2359    3   7771
2016      5472  2635  1798    0   9905
All      17725  6483  5708  294  30210
n==Taxa de default por país ==
          size   mean
Country              
EE       17725  0.425
ES        6483  0.775
FI        5708  0.732
SK         294  0.915

== Contratos por cliente ==
Clientes distintos: 20522
1     14988
2      3408
3      1201
4       507
5       184
6       106
7        49
8        27
9        12
10        9
Name: count, dtype: int64
% de contratos por cliente com 1 contrato: 0.504

Clientes distintos no treino e no teste:  1874


In [10]:
base = base[base['Country'] != 'SK'].copy()

treino_cli = set(base.loc[base['ano'] <= 2015, 'UserName'])
treino = base[base['ano'] <= 2015]
teste = base[(base['ano'] == 2016) & ~base['UserName'].isin(treino_cli)]

print('Treino:', len(treino), '| default', round(treino['default'].mean(), 3))
print('Teste: ', len(teste), '| default', round(teste['default'].mean(), 3))
print('Teste por país:\n', teste['Country'].value_counts())

Treino: 20011 | default 0.524
Teste:  7311 | default 0.671
Teste por país:
 Country
EE    3642
ES    2226
FI    1443
Name: count, dtype: int64
